# El yazısı veri seti: keşifsel veri analizi (EDA)

Veri seti: Malaysian *Potential Dysgraphia Handwriting Dataset of School-Age Children* (Mendeley Data, DOI 10.17632/39hr8dx76p.1).
Bu notebook yalnızca **okur ve ölçer**: veriyi değiştirmez, augmentation uygulamaz, model eğitmez. Çizim modülüne dokunmaz.
Bulguların özeti: `docs/handwriting_dataset_analysis.md`.

Kural: dosyalardan doğrulanan bilgi "doğrulandı", yalnızca veri seti yayınından (web) gelen bilgi "yayında yazıyor" diye ayrılır;
hiçbiri yoksa "bilinmiyor" yazılır.

In [1]:
import hashlib, random, re
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage as ndi

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CLAUDE.md").exists())
RAW = ROOT / "data" / "handwriting" / "raw"
SEED = 42
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
print("repo:", ROOT.name, "| veri:", RAW.relative_to(ROOT), "| var mı:", RAW.exists())

repo: child-drawing-analysis | veri: data\handwriting\raw | var mı: True


## 1. Klasör yapısı ve dosya formatları

In [2]:
files = sorted(p for p in RAW.rglob("*") if p.is_file())
dirs = sorted(p for p in RAW.rglob("*") if p.is_dir())
print("Klasörler:", [str(d.relative_to(RAW)) for d in dirs])
print("Toplam dosya:", len(files))
print("Uzantılar:", pd.Series([f.suffix.lower() for f in files]).value_counts().to_dict())
print("jpg dışı dosya (README, CSV, metadata vb.):", [f.name for f in files if f.suffix.lower() != ".jpg"] or "YOK")
print("Alt klasör derinliği:", {len(f.relative_to(RAW).parts) for f in files})

Klasörler: ['Low Potential Dysgraphia', 'Potential Dysgraphia']
Toplam dosya: 249
Uzantılar: {'.jpg': 249}
jpg dışı dosya (README, CSV, metadata vb.): YOK
Alt klasör derinliği: {2}


## 2. Görüntü özellikleri, sınıf dağılımı

In [3]:
def basic_info(p):
    im = Image.open(p); a = np.asarray(im)
    return dict(sinif=p.parent.name, dosya=p.name, no=int(re.search(r"\((\d+)\)", p.name).group(1)),
                w=im.width, h=im.height, mod=im.mode, bicim=im.format, dpi=im.info.get("dpi"),
                exif_alani=len(im.getexif()), icc=bool(im.info.get("icc_profile")), bayt=p.stat().st_size,
                sha256=hashlib.sha256(p.read_bytes()).hexdigest(),
                benzersiz_deger=len(np.unique(a)), uc_deger_orani=float(((a < 8) | (a > 247)).mean()),
                beyaz=float((a > 127).mean()), yol=str(p))
df = pd.DataFrame([basic_info(p) for p in files])
df["oran"] = df.w / df.h
df["kisa"] = df.sinif.map({"Low Potential Dysgraphia": "LPD", "Potential Dysgraphia": "PD"})
print(df.sinif.value_counts().rename("adet").to_frame().assign(oran=lambda d: (d.adet / d.adet.sum()).round(3)))
print("\nMod / biçim:", df.groupby(["mod", "bicim"]).size().to_dict())
print("DPI bilgisi olan:", df.dpi.notna().sum(), "| EXIF alanı olan:", (df.exif_alani > 0).sum(), "| ICC profili olan:", df.icc.sum())
print("Dosya adı numaraları sınıf içinde 1..N ardışık mı:",
      {s: sorted(g.no) == list(range(1, len(g) + 1)) for s, g in df.groupby("sinif")})

                          adet   oran
sinif                                
Low Potential Dysgraphia   135  0.542
Potential Dysgraphia       114  0.458

Mod / biçim: {('L', 'JPEG'): 249}
DPI bilgisi olan: 0 | EXIF alanı olan: 0 | ICC profili olan: 0
Dosya adı numaraları sınıf içinde 1..N ardışık mı: {'Low Potential Dysgraphia': True, 'Potential Dysgraphia': True}


In [4]:
print(df[["w", "h", "oran", "bayt"]].describe().round(1))
print("\nSınıfa göre medyan:")
print(df.groupby("kisa")[["w", "h", "oran", "bayt", "beyaz"]].median().round(3))
print("\nBenzersiz (w,h) çifti:", df.groupby(["w", "h"]).ngroups, "/", len(df))
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
for k, g in df.groupby("kisa"):
    ax[0].hist(g.w, bins=25, alpha=.6, label=k); ax[1].hist(g.h, bins=25, alpha=.6, label=k); ax[2].hist(g.oran, bins=25, alpha=.6, label=k)
for a, t in zip(ax, ["genişlik (px)", "yükseklik (px)", "en/boy oranı"]): a.set_title(t); a.legend()
plt.tight_layout(); plt.show()

            w      h   oran      bayt
count   249.0  249.0  249.0     249.0
mean   1090.6  173.9    7.0   59870.8
std     244.6   78.9    2.2   38817.0
min     396.0   55.0    3.0    9561.0
25%     918.0  102.0    5.3   30099.0
50%    1204.0  164.0    6.9   48833.0
75%    1258.0  230.0    8.1   82704.0
max    1434.0  448.0   14.2  206523.0

Sınıfa göre medyan:
           w      h   oran     bayt  beyaz
kisa                                      
LPD   1157.0  159.0  7.040  41758.0  0.074
PD    1231.0  177.5  6.714  62587.0  0.107

Benzersiz (w,h) çifti: 246 / 249


## 3. Metadata: öğrenci ID, yaş, sınıf, yazma görevi
Veri setinde ayrı bir metadata dosyası var mı, dosya adı bilgi taşıyor mu?

In [5]:
print("Dosya adı kalıpları:", df.dosya.str.replace(r"\d+", "N", regex=True).value_counts().to_dict())
print("EXIF/DPI alanı olan dosya:", int(((df.exif_alani > 0) | df.dpi.notna()).sum()))
com = sum(1 for p in df.yol if Image.open(p).info.get("comment"))
print("JPEG yorum alanı olan dosya:", com)
print("\nSonuç: öğrenci ID, yaş, sınıf (grade), cinsiyet, okul, yazma görevi için dosyalarda KAYIT YOK.")
print("Tek etiket: klasör adı (iki sınıf). Dosya numarası sınıf içinde sıra numarası; öğrenci ile eşleşmesi doğrulanamıyor.")

Dosya adı kalıpları: {'LPD (N).jpg': 135, 'PD (N).jpg': 114}
EXIF/DPI alanı olan dosya: 0
JPEG yorum alanı olan dosya: 0

Sonuç: öğrenci ID, yaş, sınıf (grade), cinsiyet, okul, yazma görevi için dosyalarda KAYIT YOK.
Tek etiket: klasör adı (iki sınıf). Dosya numarası sınıf içinde sıra numarası; öğrenci ile eşleşmesi doğrulanamıyor.


## 4. Ön işleme durumu: orijinal mi, kırpılmış mı, işlenmiş mi?

In [6]:
print("Benzersiz piksel değeri sayısı (min/medyan/maks):", df.benzersiz_deger.min(), df.benzersiz_deger.median(), df.benzersiz_deger.max())
print("Pikselin <8 ya da >247 olma oranı (ortalama):", round(df.uc_deger_orani.mean(), 6))
print("Beyaz (>127) piksel oranı: medyan", round(df.beyaz.median(), 3), "| min", round(df.beyaz.min(), 3), "| maks", round(df.beyaz.max(), 3))

def margins(p):
    a = np.asarray(Image.open(p)) > 127
    ys, xs = np.where(a)
    return dict(sol=xs.min(), sag=a.shape[1] - 1 - xs.max(), ust=ys.min(), alt=a.shape[0] - 1 - ys.max(),
                kenar_beyaz=float(np.mean([a[0].mean(), a[-1].mean(), a[:, 0].mean(), a[:, -1].mean()])))
mg = pd.DataFrame([margins(p) for p in df.yol])
print("\nKırpma payı (en yakın beyaz pikselin kenara uzaklığı, px):")
print(mg[["sol", "sag", "ust", "alt"]].describe().round(1))
print("Kenarında >%5 beyaz olan görüntü:", int((mg.kenar_beyaz > .05).sum()))
df = pd.concat([df, mg.add_prefix("pay_")], axis=1)
print("\nYorum: değerler ikili (JPEG sıkıştırma gürültüsüyle ≤ 20 gri düzeyi), yazı beyaz / arka plan siyah (ters çevrilmiş),")
print("görüntüler yazı bölgesine sıkı kırpılmış (kenar payı birkaç piksel). Yani orijinal tarama/fotoğraf DEĞİL, işlenmiş.")

Benzersiz piksel değeri sayısı (min/medyan/maks): 15 17.0 20
Pikselin <8 ya da >247 olma oranı (ortalama): 0.999988
Beyaz (>127) piksel oranı: medyan 0.083 | min 0.046 | maks 0.168

Kırpma payı (en yakın beyaz pikselin kenara uzaklığı, px):
         sol    sag    ust    alt
count  249.0  249.0  249.0  249.0
mean     5.0    4.1    9.4   10.9
std      5.2    6.4    6.8    4.9
min      0.0    0.0    0.0    0.0
25%      0.0    0.0    3.0    7.0
50%      4.0    0.0   10.0   11.0
75%      8.0    9.0   14.0   14.0
max     21.0   25.0   28.0   28.0
Kenarında >%5 beyaz olan görüntü: 9

Yorum: değerler ikili (JPEG sıkıştırma gürültüsüyle ≤ 20 gri düzeyi), yazı beyaz / arka plan siyah (ters çevrilmiş),
görüntüler yazı bölgesine sıkı kırpılmış (kenar payı birkaç piksel). Yani orijinal tarama/fotoğraf DEĞİL, işlenmiş.


## 5. Çizgili kâğıt: satır çizgileri görüntüde duruyor mu?
Yatay yönde uzun açma (opening) ile uzun beyaz çizgiler bulunur. Bu hem satır sayısı için bir yardımcı, hem de ölçümlerin
kâğıttan mı çocuktan mı geldiğini ayırmak için gerekli.

In [7]:
def load_bin(p): return np.asarray(Image.open(p)) > 127

def find_rules(b, frac=0.2):
    h, w = b.shape
    opened = ndi.binary_opening(b, structure=np.ones((1, max(15, int(frac * w)))))
    cov = opened.sum(1) / w
    rows = np.where(cov > 0.15)[0]
    groups = np.split(rows, np.where(np.diff(rows) > 2)[0] + 1) if len(rows) else []
    rules = []
    for g in groups:
        if len(g) == 0: continue
        m = np.zeros_like(opened); m[g[0]:g[-1] + 1] = opened[g[0]:g[-1] + 1]
        ys, xs = np.nonzero(m)
        slope = np.polyfit(xs, ys, 1)[0] if np.ptp(xs) > 20 else np.nan
        rules.append(dict(y=float(ys.mean()), kalinlik=len(g), kapsama=float(m.any(0).mean()), aci=float(np.degrees(np.arctan(slope)))))
    return opened, rules

rule_rows = []
for i, r in df.iterrows():
    b = load_bin(r.yol); opened, rules = find_rules(b)
    ys = [x["y"] for x in rules]
    rule_rows.append(dict(cizgi_sayisi=len(rules), cizgi_aralik=float(np.median(np.diff(ys))) if len(ys) > 1 else np.nan,
                          cizgi_aci_medyan=float(np.nanmedian([x["aci"] for x in rules])) if rules else np.nan,
                          cizgi_kapsama=float(np.mean([x["kapsama"] for x in rules])) if rules else np.nan,
                          cizgi_kalinlik=float(np.mean([x["kalinlik"] for x in rules])) if rules else np.nan,
                          bilesen_sayisi_ham=int(ndi.label(b, structure=np.ones((3, 3)))[1])))
df = pd.concat([df, pd.DataFrame(rule_rows)], axis=1)
print("Çizgi sayısı dağılımı (sınıfa göre):"); print(pd.crosstab(df.cizgi_sayisi, df.kisa))
print("\nHiç çizgisi bulunamayan görüntü:", int((df.cizgi_sayisi == 0).sum()))
print("Çizgi açısı (derece): medyan %.2f | %%5-%%95: %.2f / %.2f | |açı|>1°: %d görüntü" % (
    df.cizgi_aci_medyan.median(), df.cizgi_aci_medyan.quantile(.05), df.cizgi_aci_medyan.quantile(.95), (df.cizgi_aci_medyan.abs() > 1).sum()))
print("Çizgi aralığı (px) medyan:", df.cizgi_aralik.median(), "| min/maks:", df.cizgi_aralik.min(), df.cizgi_aralik.max())

Çizgi sayısı dağılımı (sınıfa göre):
kisa          LPD  PD
cizgi_sayisi         
1              45  11
2              55  44
3              30  27
4               3  23
5               2   9

Hiç çizgisi bulunamayan görüntü: 0
Çizgi açısı (derece): medyan -0.03 | %5-%95: -0.24 / 0.23 | |açı|>1°: 0 görüntü
Çizgi aralığı (px) medyan: 73.4470826083626 | min/maks: 38.70190713533829 212.0


## 6. Her sınıftan rastgele 10 örnek (seed 42)

In [8]:
rng = random.Random(SEED)
for k, g in df.groupby("sinif"):
    pick = rng.sample(list(g.index), 10)
    fig, axs = plt.subplots(10, 1, figsize=(10, 15))
    for a, i in zip(axs, pick):
        a.imshow(Image.open(df.yol[i]), cmap="gray"); a.set_title(f"{df.dosya[i]}  {df.w[i]}x{df.h[i]}", fontsize=7); a.axis("off")
    fig.suptitle(k, y=1.0); plt.tight_layout(); plt.show()

## 7. Görev (yazılan cümle) yapısı
Dosyalarda görev bilgisi yok. Görüntülerden görülen: birkaç sabit cümlenin kopyalanması. Çizgi sayısıyla gruplayıp
sayıları kontrol ediyoruz. Bu OCR değil, **görsel kontrol**; cümle kimliği doğrulanmış bir etiket değildir.
Görsel kontrolde en az üç farklı cümle görüldü ("Baju itu baru dibeli oleh emak." tek satır, "Burung itu berada di dalam sangkar..." iki satır,
"Saya suka bermain basikal..." üç–dört satır); ama çizgi sayısı bunları ayırmıyor (aşağıdaki tablo ve görseller).

In [9]:
print(pd.crosstab(df.cizgi_sayisi, df.kisa, margins=True))
print("\nÇizgi sayısına göre medyan boyut:")
print(df.groupby("cizgi_sayisi")[["w", "h"]].median())
for n in sorted(df.cizgi_sayisi.unique()):
    g = df[df.cizgi_sayisi == n]
    if len(g) < 3: continue
    pick = random.Random(SEED).sample(list(g.index), min(6, len(g)))
    fig, axs = plt.subplots(len(pick), 1, figsize=(9, 1.6 * len(pick)))
    for a, i in zip(np.atleast_1d(axs), pick): a.imshow(Image.open(df.yol[i]), cmap="gray"); a.set_title(f"çizgi={n} | {df.dosya[i]}", fontsize=7); a.axis("off")
    plt.tight_layout(); plt.show()

kisa          LPD   PD  All
cizgi_sayisi               
1              45   11   56
2              55   44   99
3              30   27   57
4               3   23   26
5               2    9   11
All           135  114  249

Çizgi sayısına göre medyan boyut:
                   w      h
cizgi_sayisi               
1              739.0   90.0
2             1211.0  159.0
3             1247.0  216.0
4             1262.0  273.0
5             1282.0  338.0


## 8. Tekrarlanan görüntüler ve veri sızıntısı riski
1) Birebir kopya (SHA-256). 2) Yaklaşık kopya: görüntüler 32x256'ya yeniden boyutlanıp normalize korelasyon.
Aynı cümlenin farklı çocuklarca kopyalanması da yüksek benzerlik verir; bu yüzden en benzer çiftler **gözle** kontrol edilir.

In [10]:
print("Birebir kopya (SHA-256 çakışması):", int(df.sha256.duplicated(keep=False).sum()), "görüntü")
def small(p):
    im = Image.open(p).convert("L").resize((256, 32), Image.BILINEAR)
    a = np.asarray(im, dtype=np.float32); a = (a - a.mean()) / (a.std() + 1e-6); return a.ravel()
X = np.stack([small(p) for p in df.yol]); C = X @ X.T / X.shape[1]; np.fill_diagonal(C, -1)
nn = C.argmax(1); df["en_yakin"] = nn; df["en_yakin_kor"] = C.max(1)
df["en_yakin_ayni_sinif"] = df.sinif.values == df.sinif.values[nn]
print("En yakın komşu korelasyonu: medyan %.3f | %%95: %.3f | maks: %.3f" % (df.en_yakin_kor.median(), df.en_yakin_kor.quantile(.95), df.en_yakin_kor.max()))
print("En yakın komşunun aynı sınıftan olma oranı:", round(df.en_yakin_ayni_sinif.mean(), 3))
top = df.sort_values("en_yakin_kor", ascending=False).drop_duplicates("en_yakin_kor").head(6)
print(top[["dosya", "en_yakin_kor"]].assign(komsu=[df.dosya[j] for j in top.en_yakin]).to_string())
fig, axs = plt.subplots(len(top), 2, figsize=(12, 2.0 * len(top)))
for k, (i, r) in enumerate(top.iterrows()):
    for c, j in enumerate([i, int(r.en_yakin)]):
        axs[k, c].imshow(Image.open(df.yol[j]), cmap="gray"); axs[k, c].set_title(f"{df.dosya[j]} (r={r.en_yakin_kor:.2f})", fontsize=7); axs[k, c].axis("off")
plt.tight_layout(); plt.show()

Birebir kopya (SHA-256 çakışması): 0 görüntü
En yakın komşu korelasyonu: medyan 0.404 | %95: 0.585 | maks: 0.656
En yakın komşunun aynı sınıftan olma oranı: 0.707
             dosya  en_yakin_kor          komsu
29   LPD (125).jpg      0.656126   LPD (80).jpg
100   LPD (68).jpg      0.639821   LPD (80).jpg
19   LPD (116).jpg      0.616024   LPD (65).jpg
36   LPD (131).jpg      0.611285  LPD (122).jpg
94    LPD (62).jpg      0.605861   LPD (68).jpg
9    LPD (107).jpg      0.593352   LPD (47).jpg


In [11]:
same = df[df.duplicated(["w", "h"], keep=False)].sort_values(["w", "h"])
print("Aynı (w,h) boyutundaki görüntüler:"); print(same[["dosya", "w", "h"]].to_string())
for (w, h), g in same.groupby(["w", "h"]):
    a, b = [np.asarray(Image.open(p), dtype=float) for p in g.yol.iloc[:2]]
    print(f"{w}x{h}: piksel farkı ortalama {np.abs(a - b).mean():.1f} (0 = özdeş)")

Aynı (w,h) boyutundaki görüntüler:
             dosya     w    h
92    LPD (60).jpg  1237  167
107   LPD (74).jpg  1237  167
3    LPD (101).jpg  1250  165
6    LPD (104).jpg  1250  165
175    PD (32).jpg  1269  189
194     PD (5).jpg  1269  189
1237x167: piksel farkı ortalama 40.0 (0 = özdeş)
1250x165: piksel farkı ortalama 28.6 (0 = özdeş)
1269x189: piksel farkı ortalama 47.2 (0 = özdeş)


### Sızıntı riski
Öğrenci kimliği dosyalarda yok; aynı çocuğun birden çok görüntüsünün aynı bölüme düşüp düşmeyeceği **bu veriyle denetlenemez**.
Yayında yazılanla (83 katılımcı, 249 görüntü) birlikte okununca katılımcı başına 3 görüntü beklenir (249 / 83 = 3).
Aşağıda çizgi sayısı gruplarının büyüklüğü bu beklentiyle karşılaştırılır. Beklenti tutmazsa çizgi sayısı görev ya da çocuk için
**geçerli bir vekil değildir** (kâğıttaki çizgi sayısı yazılan satır sayısına eşit değil; görsel kontrolde 5 çizgili kâğıda 2–4 satır yazılmış örnekler var).

In [12]:
vc = df.cizgi_sayisi.value_counts().sort_index()
print(vc.to_dict(), "| toplam:", vc.sum(), "| 249/83 =", 249 / 83)
print("Çizgi sayısı grupları 83'er görüntü değil → çizgi sayısı görev/çocuk eşlemesi için KULLANILAMAZ; çocuk gruplaması bu veriyle kurulamadı.")
print("Görüntü bazlı rastgele %80/%20 bölmede, 3 görüntülü bir çocuğun üçünün de aynı tarafa düşme olasılığı:",
      round(0.8**3 + 0.2**3, 3), "(yani %", round(100 * (1 - 0.8**3 - 0.2**3), 1), "ihtimalle çocuk iki bölüme dağılır)")

{1: 56, 2: 99, 3: 57, 4: 26, 5: 11} | toplam: 249 | 249/83 = 3.0
Çizgi sayısı grupları 83'er görüntü değil → çizgi sayısı görev/çocuk eşlemesi için KULLANILAMAZ; çocuk gruplaması bu veriyle kurulamadı.
Görüntü bazlı rastgele %80/%20 bölmede, 3 görüntülü bir çocuğun üçünün de aynı tarafa düşme olasılığı: 0.52 (yani % 48.0 ihtimalle çocuk iki bölüme dağılır)


## 9. Ölçüm uygulanabilirliği
Çizgi (kural) piksellerini çıkarıp kalan mürekkep üzerinde yaklaşık ölçümler denenir. **Doğruluk için referans etiket yok**; amaç
ölçümün yapılabilir olup olmadığını ve neyle karıştığını görmek. Sınıf farkı burada iddia edilmez.

In [13]:
def ink_without_rules(b):
    opened, rules = find_rules(b)
    kill = ndi.binary_dilation(opened, structure=np.ones((3, 1)))
    return b & ~kill, rules

def measure(b):
    ink, rules = ink_without_rules(b); h, w = b.shape
    lab, n = ndi.label(ink, structure=np.ones((3, 3)))
    sl = ndi.find_objects(lab); sizes = ndi.sum(ink, lab, range(1, n + 1))
    keep = [s for s, z in zip(sl, sizes) if z >= 12]
    hs = np.array([s[0].stop - s[0].start for s in keep]); ws = np.array([s[1].stop - s[1].start for s in keep])
    ys = [r["y"] for r in rules]; sp = np.median(np.diff(ys)) if len(ys) > 1 else np.nan
    edges = [0] + [y + 0.3 * sp for y in ys[:-1]] + [h] if len(ys) > 1 else [0, h]
    gaps = []
    for a_, b_ in zip(edges[:-1], edges[1:]):
        band = ink[int(a_):int(b_)]
        if band.size == 0: continue
        occ = np.where(band.any(0))[0]
        if len(occ) > 1: gaps += list(np.diff(occ)[np.diff(occ) > 1] - 1)
    gaps = np.array(gaps)
    med_h = float(np.median(hs)) if len(hs) else np.nan
    return dict(bilesen_cizgisiz=len(keep), bilesen_h_medyan=med_h, bilesen_w_medyan=float(np.median(ws)) if len(ws) else np.nan,
                yazi_h_aralik_orani=med_h / sp if sp == sp else np.nan, bosluk_sayisi=len(gaps),
                bosluk_medyan=float(np.median(gaps)) if len(gaps) else np.nan,
                genis_bosluk=int((gaps > max(8, 0.6 * med_h)).sum()) if len(gaps) and med_h == med_h else 0)
M = pd.DataFrame([measure(load_bin(p)) for p in df.yol]); df = pd.concat([df, M], axis=1)
print(df[["bilesen_sayisi_ham", "bilesen_cizgisiz", "bilesen_h_medyan", "yazi_h_aralik_orani", "bosluk_sayisi", "genis_bosluk"]].describe().round(2))
print("\nÇizgi çıkarılınca bileşen sayısı ham→çizgisiz (medyan):", df.bilesen_sayisi_ham.median(), "→", df.bilesen_cizgisiz.median())

       bilesen_sayisi_ham  bilesen_cizgisiz  bilesen_h_medyan  yazi_h_aralik_orani  bosluk_sayisi  genis_bosluk
count              249.00            249.00            249.00               193.00         249.00        249.00
mean                74.85             74.57             25.26                 0.36          60.47         14.85
std                 65.13             42.27              9.82                 0.14          32.51          8.92
min                  9.00             21.00             10.00                 0.15           8.00          0.00
25%                 30.00             35.00             18.00                 0.25          31.00          8.00
50%                 57.00             64.00             22.00                 0.31          53.00         13.00
75%                104.00            105.00             31.00                 0.46          84.00         20.00
max                375.00            233.00             54.00                 0.70         161.00       

### Satır eğimi: çizginin eğimi ve yazının eğimi
Çizgi açısı kâğıdın (ve görüntü işlemenin) eğimini verir. Yazı satırı eğimi için çizgisiz mürekkepte izdüşüm profili taranır.

In [14]:
from skimage.transform import rotate
def text_angle(ink, rng_deg=np.arange(-6, 6.01, 0.5)):
    sm = ink[:, ::2].astype(float)
    best, bs = 0, -1
    for ang in rng_deg:
        r = rotate(sm, ang, order=0, preserve_range=True)
        s = r.sum(1).var()
        if s > bs: best, bs = ang, s
    return best
ta = []
for p in df.yol:
    ink, _ = ink_without_rules(load_bin(p)); ta.append(text_angle(ink))
df["yazi_aci"] = ta
print("Çizgi açısı (°): ", df.cizgi_aci_medyan.describe().round(2).to_dict())
print("Yazı satırı açısı (°, izdüşüm taraması):", df.yazi_aci.describe().round(2).to_dict())
print("Yazı-çizgi açı farkı mutlak medyan: %.2f°" % (df.yazi_aci - df.cizgi_aci_medyan).abs().median())
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].hist(df.cizgi_aci_medyan.dropna(), bins=30); ax[0].set_title("çizgi açısı (°)")
ax[1].hist(df.yazi_aci, bins=25); ax[1].set_title("yazı açısı, izdüşüm (°)"); plt.tight_layout(); plt.show()

Çizgi açısı (°):  {'count': 249.0, 'mean': -0.03, 'std': 0.13, 'min': -0.37, '25%': -0.1, '50%': -0.03, '75%': 0.03, 'max': 0.35}
Yazı satırı açısı (°, izdüşüm taraması): {'count': 249.0, 'mean': -0.09, 'std': 0.72, 'min': -4.5, '25%': -0.5, '50%': 0.0, '75%': 0.0, 'max': 2.0}
Yazı-çizgi açı farkı mutlak medyan: 0.27°


### Ölçümlerin karıştığı şeyler (ölçek ve görüntü biçimi)
Görüntülerde fiziksel ölçek (DPI) yok; piksel boyutları kırpma ve yeniden boyutlandırmaya bağlı. Sınıf medyanları yalnızca
**karışıklık kontrolü** için: sınıflar arasında boyut ya da mürekkep farkı varsa ölçümler yazı kalitesinden çok görüntü özelliğini yansıtabilir.

In [15]:
cols = ["w", "h", "cizgi_sayisi", "cizgi_aralik", "cizgi_kalinlik", "beyaz", "bilesen_h_medyan", "yazi_h_aralik_orani", "genis_bosluk", "yazi_aci"]
print(df.groupby("kisa")[cols].median().round(3).T)
print("\nÇizgi sayısı x sınıf (görev karışıklığı):")
print(df.groupby(["cizgi_sayisi", "kisa"]).size().unstack(fill_value=0))
print("\nÇizgi aralığı ile bileşen yüksekliği korelasyonu (ölçek bağımlılığı):",
      round(df[["cizgi_aralik", "bilesen_h_medyan"]].corr().iloc[0, 1], 2))

kisa                      LPD        PD
w                    1157.000  1231.000
h                     159.000   177.500
cizgi_sayisi            2.000     3.000
cizgi_aralik           76.417    72.922
cizgi_kalinlik          2.500     3.000
beyaz                   0.074     0.107
bilesen_h_medyan       20.000    31.000
yazi_h_aralik_orani     0.267     0.441
genis_bosluk           11.000    16.000
yazi_aci                0.000     0.000

Çizgi sayısı x sınıf (görev karışıklığı):
kisa          LPD  PD
cizgi_sayisi         
1              45  11
2              55  44
3              30  27
4               3  23
5               2   9

Çizgi aralığı ile bileşen yüksekliği korelasyonu (ölçek bağımlılığı): 0.23


In [16]:
df.drop(columns=["yol"]).to_csv(ROOT / "data" / "handwriting" / "eda_ozellikler.csv", index=False, encoding="utf-8")
print("Yazıldı: data/handwriting/eda_ozellikler.csv (git'e girmez, data/ ignore). raw/ klasörüne dokunulmadı.")

Yazıldı: data/handwriting/eda_ozellikler.csv (git'e girmez, data/ ignore). raw/ klasörüne dokunulmadı.
